# v4 两个复杂样本的更长超时诊断

日期：2026-11-02（Asia/Shanghai）。批次：`formal_30_v4_timeout_diagnostic_v1`。本记录于首次模型调用前建立，当前无诊断结果。

原完整 30 项开发回归及一次同条件恢复后仍为 25 通过、3 评估失败、2 执行超时。2026-10-30 的两项重试都在 generation 请求约 241 秒处中断，符合当时 240 秒上游读超时上限；不能区分服务端排队、生成耗时或网络读等待。

旧证据：`formal_30_v4_full_regression_results_20261030_215511` 与 `formal_30_v4_full_regression_results_20261030_231238`，各 359 个文件校验通过。两次记录均保留，新诊断不改写其 25/30 分数。

## 预先固定的诊断条件

- 样本：TaskManager、MonitoringDashboard（原 Vue 输入、原顺序，各自重新生成）。
- 保持 qwen/qwen-plus、温度 0.2、v4 的 SSM／Prompt／验证器／repair 实现及最多 3 轮 repair。
- 上游 `LLM_TIMEOUT` 从 240 增至 600 秒；客户端 `MIGRATION_GENERATION_TIMEOUT` 和 `MIGRATION_REPAIR_TIMEOUT` 从 300 增至 660 秒。
- 通过专用 launcher 设置新子进程环境，服务器必须真正停止旧进程并重启。JSON 中环境值只是证据元数据，普通批处理不会读取它来改变超时。仓库全局默认值保持。
- 输出和 checkpoint 使用独立的 `formal_30_v4_timeout_diagnostic_v1` 路径，不加 --force，不以新结果回填原 30 项结果。

由于超时条件和模型随机采样变化，本次若成功，只能说明更长超时条件下完成了本次运行，不能仅凭一次结果证明原失败唯一由时间不足导致。原 3 项评估失败仍列为后续系统修复，v5 继续等待。

## 运行和后续记录

协议：`data/experiments/protocols/formal_30_v4_timeout_diagnostic_v1_frozen_20261102/PROTOCOL.md`。

先停止旧服务器，在窗口 A 设置环境中的 LLM_API_KEY 后运行：`python -m migration_pipeline.run_formal_30_v4_timeout_diagnostic --server`。密钥不写入文件。

窗口 B 先执行 `python -m migration_pipeline.run_formal_30_v4_timeout_diagnostic --dry-run`，再执行一次 `python -m migration_pipeline.run_formal_30_v4_timeout_diagnostic --execute`。单次上游请求可能接近 10 分钟，整批耗时更长。执行完成或报错后先冻结原始结果，再追加通过情况、失败阶段、耗时、轮数和已记录的 token；异常请求用量可能缺失，需单列。

首次调用前的单元测试、旧代码字节对照和 dry-run 封存在协议目录中。服务器重启与实际进程环境需要操作者确认，文件哈希不能单独证明。

2026-11-02 预运行验证完成：229/229 单元测试通过（含独立子进程实际加载 600／660／660 秒配置的测试）；旧重试快照 359 文件校验通过，旧运行基线 live 对照通过。新协议封存 249 个 payload，共 256 个哈希校验项，live 校验及两样本 dry-run 均通过。此时模型请求尚未发生，运行 summary 尚不存在；下面另记实际运行结果。

## 诊断结果：执行完成，迁移仍未通过（2026-11-02）

运行 17:11:02—17:29:33 +08:00。两项均完成，无执行超时；自动结果为 **0/2 通过、2 项评估失败**，各消耗 3 个 repair attempt。原始结果先冻结于 `data/experiments/snapshots/formal_30_v4_timeout_diagnostic_v1_results_20261102_173627`，272 个文件校验通过，协议 256 项 live 校验通过。

| 样本 | 总耗时 | 确定性 repair 轮 | LLM repair 轮 | 停止原因 |
| --- | ---: | ---: | ---: | --- |
| TaskManager | 250.515 秒 | 3 | 0 | max_repair_rounds_reached |
| MonitoringDashboard | 860.859 秒 | 1 | 2 | repair_event_contract_regression |

**TaskManager**：三轮分别做结构／事件语义归一化、事件处理器归一化、再次恢复 self-click／拖拽语义。repair 阶段只要确定性计划改变代码便直接返回，每次都会被 orchestrator 计为一轮；因此本次根本没有 LLM repair。最终仍有 9 类静态错误，含内联表达式、事件语法／payload、循环别名／键、computed 方法、隐藏依赖和原地数组修改；初始 DOM 相似度 0.9889 不能替代这些检查。重复归一化是规则交互审查线索，尚未证明存在无限循环。

**MonitoringDashboard**：第 2 轮后有 3 类静态错误、DOM 相似度 0.9605；第 3 轮新引入 missing_event_bindings，触发事件契约保护门，另出现 initData 冲突及 this.initData is not a function 渲染错误。保护门的实际行为是停止并拒绝计为通过，没有回滚；final_file_path 仍是第 3 轮。第 2 轮也未通过静态校验，不能当作替代成功产物。

生成 204,063、repair 162,383、总计 366,446 tokens。4 轮确定性处理使用 0 个 LLM token，但消耗 attempt 预算。单样本总耗时并非单次 HTTP 请求耗时，不能据此证明每项必须超过 240 秒，也不能把执行不超时等同于迁移通过。旧 30 项结果继续保留为 25 通过、3 评估失败、2 超时，不回填本诊断结果。

## 后续开发修复优先级与追溯

下一修复版优先核查确定性处理的幂等性、不同规则的相互覆盖、轮数预算及 LLM 调度，并把确定性步骤数与 LLM 调用数分开记录；预算定义变化必须在新版本中预先登记，保留当前 3 轮原结果。随后核查 repair 引入 initData 冲突和事件退化时的拒收／候选保留机制，并连同原 BudgetThreshold、TipSplitCalculator、LocalizationWorkspace 三项失败做冻结回放。新版本先本地验证，再做针对性模型运行和完整 30 项回归；v5 仍等待。

分析报告：`data/experiments/analysis/formal_30_v4_timeout_diagnostic_v1_20261102_173627/REPORT.md`，逐轮来源、归一化标记、错误明细、最终产物哈希见同目录 `summary.json`。复现：`python -m migration_pipeline.analyze_formal_30_v4_timeout_diagnostic --snapshot data/experiments/snapshots/formal_30_v4_timeout_diagnostic_v1_results_20261102_173627`。本次没有改动 Prompt、运行规则、轮数定义或生成产物，仅封存并分析用户完成的运行。